In [ ]:
# Start /analyze endpoint in Colab

!pip -q install fastapi uvicorn pyngrok python-multipart pillow transformers accelerate bitsandbytes sentencepiece nest_asyncio requests

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import io
import gc
import threading
import time
import requests
import torch
from PIL import Image
from fastapi import FastAPI, UploadFile, File, Form, Header, HTTPException
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
import uvicorn
import nest_asyncio

nest_asyncio.apply()

# Hugging Face login
try:
    from huggingface_hub import login
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        login(token=token)
except Exception:
    pass

gc.collect()
torch.cuda.empty_cache()

app = FastAPI()

MODEL_ID = "llava-hf/llava-1.5-7b-hf"
AUTH_TOKEN = None

try:
    AUTH_TOKEN = userdata.get("API_AUTH_TOKEN") or None
except Exception:
    AUTH_TOKEN = None

print("Loading model and processor...")

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID, use_fast=True)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

print("Model loaded!")

def prep_image(img: Image.Image) -> Image.Image:
    img = img.convert("RGB")
    img.thumbnail((768, 768))
    return img

@app.get("/health")
def health():
    return {"ok": True, "model_loaded": True}

@app.post("/analyze")
async def analyze(
    image: UploadFile = File(...),
    prompt: str = Form(...),
    authorization: str | None = Header(default=None),
):
    if AUTH_TOKEN:
        expected = f"Bearer {AUTH_TOKEN}"
        if authorization != expected:
            raise HTTPException(status_code=401, detail="Unauthorized")

    try:
        # Read the uploaded image
        img_bytes = await image.read()
        pil_img = prep_image(Image.open(io.BytesIO(img_bytes)))

        # Build chat input
        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt}
                ],
            }
        ]

        prompt_text = processor.apply_chat_template(
            conversation,
            add_generation_prompt=True
        )

        inputs = processor(
            text=prompt_text,
            images=pil_img,
            return_tensors="pt"
        ).to(model.device)

        # Run generation
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=96,
                do_sample=False,
                num_beams=1
            )

        # Decode the output
        text = processor.decode(out[0], skip_special_tokens=True)
        description = text.split("ASSISTANT:")[-1].strip() if "ASSISTANT:" in text else text.strip()

        # Clear GPU memory
        del inputs, out
        gc.collect()
        torch.cuda.empty_cache()

        return {"description": description}

    except Exception as e:
        gc.collect()
        torch.cuda.empty_cache()
        raise HTTPException(status_code=500, detail=str(e))

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

# Start the server
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait until the server is ready
def wait_for_server():
    for i in range(20):
        try:
            r = requests.get("http://127.0.0.1:8000/health", timeout=2)
            if r.status_code == 200:
                print("✅ Server is ready!")
                print(r.json())
                return
        except:
            pass
        time.sleep(1)
    print("Server did not start")

wait_for_server()

print("🚀 API running on http://127.0.0.1:8000")

In [ ]:
# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("cloudflared installed successfully")

In [ ]:
# Create a public tunnel
import subprocess, re, time

# Wait for the local server
time.sleep(5)

# Check server health
import requests
try:
    response = requests.get("http://localhost:8000/health", timeout=5)
    if response.status_code == 200:
        print("Server is running and healthy!")
        print(f"Health check response: {response.json()}")
    else:
        print(f"Server returned status code: {response.status_code}")
except Exception as e:
    print(f"Server not ready yet: {e}")
    print("Make sure the server cell ran successfully before running this cell.")

# Start the tunnel
p = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
print("\nStarting tunnel...")
for _ in range(120):
    line = p.stdout.readline().strip()
    if line:
        print(line)
    m = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break

if public_url:
    print(f" PUBLIC URL: {public_url}")
    print(f"ANALYZE ENDPOINT: {public_url}/analyze")
    print(f"HEALTH CHECK: {public_url}/health")

else:
    print("Failed to get public URL. Check tunnel logs above.")

In [ ]:
# Test the local endpoint
import requests

# Check local health
try:
    response = requests.get("http://localhost:8000/health", timeout=5)
    print(f"Health check: {response.status_code}")
    print(f"Response: {response.json()}")
except Exception as e:
    print(f"Error: {e}")
    print("Make sure the server is running (run the first cell first)")